In [1]:
import os
from datetime import datetime
import pytorch_lightning as pl

from config.settings import BERTConfig
from infrastructure.data_pipeline import BERTTextDataModule, WordTokenizer
from application.lightning_module import BERTPreTrainingLightning


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
def get_next_version_string(base_dir: str, name: str) -> str:
    """Calcula la siguiente versión incremental y le anexa la fecha/hora actual."""
    target_dir = os.path.join(base_dir, name)
    next_v = 0
    if os.path.exists(target_dir):
        for folder in os.listdir(target_dir):
            if folder.startswith("v"):
                try:
                    v_num = int(folder.split('_')[0][1:])
                    next_v = max(next_v, v_num + 1)
                except (ValueError, IndexError):
                    continue
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    return f"v{next_v}_{timestamp}"

In [4]:
if not os.path.exists("corpus.txt"):
    with open("corpus.txt", "w", encoding="utf-8") as f:
        f.write("This is the first sentence of the text.\n")
        f.write("Here is the second sentence.\n")
        f.write("Deep learning models are very powerful.\n")
        f.write("They require a lot of data to train properly.\n")
            
config = BERTConfig()
    
data_module = BERTTextDataModule(
    txt_file_path="corpus.txt",
    batch_size=config.batch_size,
    max_len=config.max_len,
    num_workers=7
    )

data_module.setup()

config.vocab_size = data_module.vocab_size
config.pad_idx = data_module.pad_token_id

model = BERTPreTrainingLightning(config)

version_str = get_next_version_string("tb_logs", "bert_modular")
logger = pl.loggers.TensorBoardLogger(
    save_dir="tb_logs", 
    name="bert_modular",
    version=version_str
    )

trainer = pl.Trainer(
    max_epochs=30,  
    logger=logger,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10,
    gradient_clip_val=1.0,
    gradient_clip_algorithm="norm"
    )
    
print(f"Iniciando entrenamiento modular... (Log version: {version_str})")
trainer.fit(model, datamodule=data_module)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Iniciando entrenamiento modular... (Log version: v5_2026-06-05_05-55-02)


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ bert      │ BERT             │  2.3 M │ train │     0 │
│ 1 │ mlm_head  │ Linear           │  735 K │ train │     0 │
│ 2 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 2.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.3 M                                                                                                
Total estimated model params size (MB): 9.276                                                                      
Modules in train mode: 39                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/Users/cesar/sandbox/bert_experiment/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_c
onnector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker 
initialization.

/Users/cesar/sandbox/bert_experiment/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_c
onnector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker
initialization.

/Users/cesar/sandbox/bert_experiment/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:156: 
UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. 
Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the 
closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to 
replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)

`Trainer.fit` stopped: `max_epochs=30` reached.


In [15]:
import torch
import random

# 1. Lista de frases de prueba en español
frases_prueba = [
    "El Principito es un niño",
    "Oigo por la noche.",
    "La visita al tercer planeta."
]

# Ponemos el modelo en modo evaluación
model.eval()

# Recuperamos nuestro WordTokenizer desde el data_module
tokenizer = data_module.tokenizer

print("Iniciando pruebas de enmascaramiento aleatorio con WordTokenizer...\n")
print("=" * 60)

with torch.no_grad():
    for frase in frases_prueba:
        # 2. Tokenizar la frase limpia usando .encode() de WordTokenizer
        # Esto nos devuelve una lista de enteros: ej. [12, 15, 8, 94]
        token_ids = tokenizer.encode(frase)
        
        # BERT requiere obligatoriamente [CLS] al inicio y [SEP] al final para entender el contexto
        input_ids_list = [tokenizer.cls_token_id] + token_ids + [tokenizer.sep_token_id]
        
        # Convertimos la lista manualmente en un tensor de PyTorch con dimensiones de lote (Batch)
        # Queda con dimensiones (1, longitud_secuencia)
        input_ids = torch.tensor([input_ids_list], dtype=torch.long)
        
        # Creamos los segment_ids (0 para todo el lote, ya que es una única frase)
        segment_ids = torch.zeros_like(input_ids)
        
        seq_len = input_ids.shape[1]
        
        # Si la frase es extremadamente corta, la ignoramos
        if seq_len <= 3:
            print(f"Frase ignorada por ser muy corta: {frase}")
            continue
            
        # Elegimos una posición al azar protegiendo el índice 0 ([CLS]) y el último ([SEP])
        mask_idx = random.randint(1, seq_len - 2)
        
        # Guardamos el ID original para saber qué palabra ocultamos
        original_token_id = input_ids[0, mask_idx].item()
        palabra_esperada = tokenizer.convert_ids_to_tokens([original_token_id])[0]
        
        # Aplicamos el token de máscara [MASK] en la posición elegida
        input_ids[0, mask_idx] = tokenizer.mask_token_id
        
        # Decodificamos la frase modificada para ver visualmente el [MASK]
        frase_enmascarada = tokenizer.decode(input_ids[0], skip_special_tokens=False)
        
        # 3. Pasar los tensores por el modelo entrenado
        logits = model(input_ids, segment_ids)
        
        # Extraemos las probabilidades de la palabra oculta
        mask_logits = logits[0, mask_idx, :]
        
        # Tomamos el ID con el valor más alto (la predicción del modelo)
        predicted_token_id = mask_logits.argmax(dim=-1).item()
        palabra_predicha = tokenizer.convert_ids_to_tokens([predicted_token_id])[0]
        
        # Limpiamos visualmente los tokens especiales del print del resultado
        frase_limpia_mostrar = frase_enmascarada.replace("[CLS] ", "").replace(" [SEP]", "")
        
        # Imprimir métricas de control
        print(f"Original:    {frase}")
        print(f"Enmascarada: {frase_limpia_mostrar}")
        print(f"Esperado:    [{palabra_esperada}]")
        print(f"Predicción:  [{palabra_predicha}]")
        print("-" * 60)

Iniciando pruebas de enmascaramiento aleatorio con WordTokenizer...

Original:    El Principito es un niño
Enmascarada: el principito es un [MASK]
Esperado:    [niño]
Predicción:  [principito]
------------------------------------------------------------
Original:    Oigo por la noche.
Enmascarada: oigo por la [MASK] .
Esperado:    [noche]
Predicción:  [serpiente]
------------------------------------------------------------
Original:    La visita al tercer planeta.
Enmascarada: la visita al tercer [MASK] .
Esperado:    [planeta]
Predicción:  [principito]
------------------------------------------------------------
